# HPI Prediction: ARIMA

This notebook forecasts House Price Index (HPI) for all 51 U.S. states using ARIMA, then evaluates performance on a 5-year holdout set (2020–2024).

## What is ARIMA?

**ARIMA** (AutoRegressive Integrated Moving Average) is a classical time series forecasting model defined by three parameters **(p, d, q)**:

- **AR (p)** — *AutoRegressive*: The model uses the previous *p* values of the series to predict the next value. This captures momentum — e.g., if HPI has been rising, it's likely to continue rising.
- **I (d)** — *Integrated*: The series is differenced *d* times to make it stationary (removing trends). For HPI, *d* = 1 means we model year-over-year changes rather than raw index values.
- **MA (q)** — *Moving Average*: The model uses the previous *q* forecast errors to correct its predictions. This helps the model adapt when recent predictions were off.

### Why ARIMA for HPI?

- HPI is a **univariate annual time series** with clear trends — exactly the type of data ARIMA is designed for.
- We use `pmdarima.auto_arima` to **automatically select** the best (p, d, q) order for each state via AIC minimization, so each state gets a tailored model.

**Metrics used**: MAE (Mean Absolute Error), RMSE (Root Mean Squared Error), and MAPE (Mean Absolute Percentage Error) — MAPE is the primary metric since it's scale-independent across states.

In [1]:
import sys
import warnings
from pathlib import Path

# add src/ to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np
from data_cleaning import prepare_state_hpi
from prediction import (
    split_train_test,
    run_arima_all_states,
    compute_metrics,
)
from plot_utils import (
    plot_state_forecast, plot_forecast_grid,
    plot_forecast_choropleth, plot_forecast_choropleth_animated,
    plot_metrics_bar,
)

print("Imports loaded successfully.")

Imports loaded successfully.


## 1. Load & Explore HPI Data

In [2]:
raw_dir = project_root / "data" / "raw"
df = prepare_state_hpi(raw_dir)

print(f"Shape: {df.shape}")
print(f"States: {df['Abbreviation'].nunique()}")
print(f"Year range: {df['Year'].min()} – {df['Year'].max()}")
df.head(10)

Shape: (2550, 4)
States: 51
Year range: 1975 – 2024


,Abbreviation,State,Year,HPI
0,AK,Alaska,1975,100.00
1,AK,Alaska,1976,109.22
2,AK,Alaska,1977,117.82
3,AK,Alaska,1978,129.79
4,AK,Alaska,1979,144.21
5,AK,Alaska,1980,136.60
6,AK,Alaska,1981,165.50
7,AK,Alaska,1982,206.50
8,AK,Alaska,1983,218.60
9,AK,Alaska,1984,233.42


In [3]:
# quick look at HPI distribution across states
latest_year = df["Year"].max()
latest = df[df["Year"] == latest_year].sort_values("HPI", ascending=False)
print(f"\nHPI in {latest_year} — Top 5 and Bottom 5:")
print(latest[["Abbreviation", "State", "HPI"]].head())
print("...")
print(latest[["Abbreviation", "State", "HPI"]].tail())


HPI in 2024 — Top 5 and Bottom 5:
     Abbreviation                 State      HPI
399            DC  District Of Columbia  1975.49
249            CA            California  1955.55
2399           WA            Washington  1836.58
999            MA         Massachusetts  1622.74
599            HI                Hawaii  1507.13
...
     Abbreviation          State     HPI
649            IA           Iowa  624.69
99             AL        Alabama  610.27
949            LA      Louisiana  599.87
1299           MS    Mississippi  489.84
2499           WV  West Virginia  444.40


## 2. ARIMA Forecasting (All States)

Uses `pmdarima.auto_arima` to automatically select (p,d,q) order per state. This takes ~1-3 minutes.

In [4]:
%%time
arima_forecast, arima_eval, arima_models = run_arima_all_states(
    df, forecast_years=10, test_years=5
)
print(f"ARIMA forecasts: {len(arima_forecast)} rows")
print(f"ARIMA eval: {len(arima_eval)} rows")

/Users/andrewpark/Desktop/College/2025-2026/Winter Quarter/ECE 143/ECE143_Historical_Housing_Trends/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/andrewpark/Desktop/College/2025-2026/Winter Quarter/ECE 143/ECE143_Historical_Housing_Trends/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/andrewpark/Desktop/College/2025-2026/Winter Quarter/ECE 143/ECE143_Historical_Housing_Trends/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/andrew

ARIMA forecasts: 510 rows
ARIMA eval: 255 rows
CPU times: user 12.8 s, sys: 135 ms, total: 13 s
Wall time: 13 s


## 3. Evaluate ARIMA on 2020–2024 Holdout

In [5]:
metrics = compute_metrics(arima_eval)

print("=== ARIMA Metrics (Best 10 by MAPE) ===")
print(metrics.head(10).to_string(index=False))
print(f"\nMedian MAPE: {metrics['MAPE'].median():.2f}%")
print(f"Mean MAPE: {metrics['MAPE'].mean():.2f}%")
print(f"States with MAPE < 15%: {(metrics['MAPE'] < 15).sum()}/51")

print("\n=== Worst 10 by MAPE ===")
print(metrics.tail(10).to_string(index=False))

=== ARIMA Metrics (Best 10 by MAPE) ===
Abbreviation                State    MAE   RMSE  MAPE
          DC District Of Columbia  70.31  84.75  3.59
          LA            Louisiana  42.96  51.05  7.33
          ND         North Dakota  52.46  64.97  7.95
          MN            Minnesota  86.82 102.78 10.42
          IL             Illinois  70.54  87.61 11.10
          AK               Alaska  71.02  86.96 11.50
          IA                 Iowa  70.12  85.82 11.98
          NE             Nebraska  84.25 102.27 12.00
          MD             Maryland 114.14 137.30 12.47
          WV        West Virginia  51.89  61.93 12.52

Median MAPE: 15.45%
Mean MAPE: 15.96%
States with MAPE < 15%: 22/51

=== Worst 10 by MAPE ===
Abbreviation          State    MAE   RMSE  MAPE
          RI   Rhode Island 238.55 287.15 19.72
          UT           Utah 241.13 278.08 19.82
          TN      Tennessee 157.71 188.44 19.89
          NC North Carolina 161.60 196.84 20.56
          MT        Montana 253

## 4. Visualizations

### 4a. Single State Forecast (California)

In [6]:
fig = plot_state_forecast("CA", df, arima_forecast, eval_df=arima_eval)
fig.show()

### 4b. Multi-State Forecast Grid

In [7]:
fig = plot_forecast_grid(
    df, arima_forecast, eval_df=arima_eval,
    states=["CA", "TX", "NY", "FL", "IL", "WA"],
)
fig.show()

### 4c. Forecasts for All 51 States

In [8]:
all_states = sorted(df["Abbreviation"].unique())
fig = plot_forecast_grid(
    df, arima_forecast, eval_df=arima_eval,
    states=all_states,
)
fig.show()

### 4d. Choropleth Map (Predicted HPI in 2030)

In [9]:
fig = plot_forecast_choropleth(arima_forecast, 2030)
fig.show()

### 4e. Animated Choropleth (History + Forecast)

In [10]:
fig = plot_forecast_choropleth_animated(df, arima_forecast, history_years=5)
fig.show()

### 4f. MAPE by State Bar Chart

In [11]:
fig = plot_metrics_bar(metrics, metric="MAPE", top_n=20)
fig.show()

## 5. Comprehensive Analysis

### 5a. ARIMA Model Orders Selected per State

`auto_arima` selects a different (p, d, q) order for each state. Let's examine what orders were chosen and what that tells us about each state's HPI dynamics.

In [12]:
# extract ARIMA orders for each state
order_rows = []
for state, model in arima_models.items():
    p, d, q = model.order
    order_rows.append({"Abbreviation": state, "p": p, "d": d, "q": q,
                       "Order": f"({p},{d},{q})", "AIC": round(model.aic(), 2)})
orders_df = pd.DataFrame(order_rows).sort_values("Abbreviation")

print("ARIMA Orders Selected by auto_arima:\n")
print(orders_df.to_string(index=False))

print(f"\n--- Order Distribution ---")
print(orders_df["Order"].value_counts().to_string())
print(f"\nMost common d (differencing): {orders_df['d'].mode().iloc[0]} "
      f"({(orders_df['d'] == orders_df['d'].mode().iloc[0]).sum()}/51 states)")
print(f"Mean AIC: {orders_df['AIC'].mean():.1f}")

ARIMA Orders Selected by auto_arima:

Abbreviation  p  d  q   Order    AIC
          AK  1  1  0 (1,1,0) 401.40
          AL  1  1  1 (1,1,1) 353.66
          AR  2  1  2 (2,1,2) 362.96
          AZ  1  1  1 (1,1,1) 492.39
          CA  0  1  2 (0,1,2) 546.24
          CO  2  2  1 (2,2,1) 446.10
          CT  2  1  0 (2,1,0) 416.54
          DC  0  1  1 (0,1,1) 506.35
          DE  1  1  1 (1,1,1) 390.26
          FL  0  1  2 (0,1,2) 459.42
          GA  1  1  1 (1,1,1) 397.46
          HI  2  1  1 (2,1,1) 506.61
          IA  1  1  1 (1,1,1) 338.58
          ID  1  1  0 (1,1,0) 478.34
          IL  1  1  0 (1,1,0) 395.58
          IN  1  1  1 (1,1,1) 365.15
          KS  1  1  2 (1,1,2) 333.36
          KY  1  1  2 (1,1,2) 343.67
          LA  1  1  0 (1,1,0) 366.86
          MA  1  1  1 (1,1,1) 457.48
          MD  2  1  0 (2,1,0) 433.85
          ME  1  1  1 (1,1,1) 425.68
          MI  1  1  0 (1,1,0) 391.10
          MN  1  1  0 (1,1,0) 419.48
          MO  1  1  1 (1,1,1) 352.18


### 5b. Regional Performance Analysis

States can be grouped by U.S. Census region. Do certain regions have systematically higher or lower forecast errors?

In [13]:
# define Census regions
regions = {
    "Northeast": ["CT", "ME", "MA", "NH", "RI", "VT", "NJ", "NY", "PA"],
    "Midwest": ["IL", "IN", "MI", "OH", "WI", "IA", "KS", "MN", "MO", "NE", "ND", "SD"],
    "South": ["DE", "FL", "GA", "MD", "NC", "SC", "VA", "DC", "WV",
              "AL", "KY", "MS", "TN", "AR", "LA", "OK", "TX"],
    "West": ["AZ", "CO", "ID", "MT", "NV", "NM", "UT", "WY",
             "AK", "CA", "HI", "OR", "WA"],
}
state_to_region = {s: r for r, states in regions.items() for s in states}
metrics_with_region = metrics.copy()
metrics_with_region["Region"] = metrics_with_region["Abbreviation"].map(state_to_region)

print("=== MAPE by Region ===\n")
region_stats = metrics_with_region.groupby("Region")["MAPE"].agg(["mean", "median", "min", "max", "count"])
region_stats.columns = ["Mean MAPE", "Median MAPE", "Best MAPE", "Worst MAPE", "N States"]
region_stats = region_stats.sort_values("Median MAPE")
print(region_stats.to_string())

print("\n\n=== Best & Worst State per Region ===\n")
for region in region_stats.index:
    r = metrics_with_region[metrics_with_region["Region"] == region].sort_values("MAPE")
    best = r.iloc[0]
    worst = r.iloc[-1]
    print(f"{region}:")
    print(f"  Best:  {best['Abbreviation']} ({best['State']}) — MAPE {best['MAPE']:.2f}%")
    print(f"  Worst: {worst['Abbreviation']} ({worst['State']}) — MAPE {worst['MAPE']:.2f}%")

=== MAPE by Region ===

           Mean MAPE  Median MAPE  Best MAPE  Worst MAPE  N States
Region                                                            
Midwest    12.930833       13.465       7.95       15.85        12
South      15.218824       15.420       3.59       28.00        17
West       18.967692       17.340      11.50       26.23        13
Northeast  17.030000       17.440      14.19       19.72         9


=== Best & Worst State per Region ===

Midwest:
  Best:  ND (North Dakota) — MAPE 7.95%
  Worst: SD (South Dakota) — MAPE 15.85%
South:
  Best:  DC (District Of Columbia) — MAPE 3.59%
  Worst: FL (Florida) — MAPE 28.00%
West:
  Best:  AK (Alaska) — MAPE 11.50%
  Worst: NV (Nevada) — MAPE 26.23%
Northeast:
  Best:  MA (Massachusetts) — MAPE 14.19%
  Worst: RI (Rhode Island) — MAPE 19.72%


### 5c. Error Pattern: Did the Model Consistently Under- or Over-Predict?

The 2020–2024 holdout period includes the COVID-era housing boom. Let's check whether ARIMA systematically underestimated the surge.

In [14]:
import plotly.express as px
from scipy import stats

# compute signed error per state per year
eval_merged = arima_eval.merge(
    df[["Abbreviation", "Year", "HPI"]], on=["Abbreviation", "Year"], how="left"
)
eval_merged["Signed_Error"] = eval_merged["Predicted"] - eval_merged["HPI"]
eval_merged["Pct_Error"] = (eval_merged["Signed_Error"] / eval_merged["HPI"] * 100)

# year-by-year bias
print("=== Year-by-Year Forecast Bias (Predicted − Actual) ===\n")
yearly = eval_merged.groupby("Year").agg(
    Mean_Signed_Error=("Signed_Error", "mean"),
    Median_Signed_Error=("Signed_Error", "median"),
    Mean_Pct_Error=("Pct_Error", "mean"),
    MAPE=("Pct_Error", lambda x: x.abs().mean()),
    Underpredicted=("Signed_Error", lambda x: (x < 0).sum()),
).reset_index()
yearly["Year"] = yearly["Year"].astype(int)

for _, r in yearly.iterrows():
    direction = "UNDER" if r["Mean_Signed_Error"] < 0 else "OVER"
    print(f"  {r['Year']}:  Mean Error: {r['Mean_Signed_Error']:+7.1f}  "
          f"({r['Mean_Pct_Error']:+5.1f}%)  |  MAPE: {r['MAPE']:5.1f}%  |  "
          f"{direction}-predicted {int(r['Underpredicted'])}/51 states")

print(f"\nOverall: ARIMA underpredicted in {int(yearly['Underpredicted'].mean())}/51 states on average.")
print("The bias grows each year — the model missed the accelerating post-COVID surge.\n")

# --- Scatter plot: HPI growth (2019→2024) vs MAPE ---
growth_rates = []
for state in df["Abbreviation"].unique():
    state_df = df[df["Abbreviation"] == state].sort_values("Year")
    hpi_2019 = state_df[state_df["Year"] == 2019]["HPI"].values
    hpi_2024 = state_df[state_df["Year"] == 2024]["HPI"].values
    if len(hpi_2019) > 0 and len(hpi_2024) > 0:
        pct_growth = (hpi_2024[0] - hpi_2019[0]) / hpi_2019[0] * 100
        growth_rates.append({"Abbreviation": state, "Growth_Pct": round(pct_growth, 1)})
growth_df = pd.DataFrame(growth_rates)
scatter_df = metrics.merge(growth_df, on="Abbreviation")

# OLS trendline
slope, intercept, r_value, _, _ = stats.linregress(scatter_df["Growth_Pct"], scatter_df["MAPE"])

fig = px.scatter(
    scatter_df, x="Growth_Pct", y="MAPE", text="Abbreviation",
    labels={"Growth_Pct": "HPI Growth 2019→2024 (%)", "MAPE": "MAPE (%)"},
    title=f"Forecast Error vs Housing Boom Intensity (R² = {r_value**2:.2f})",
)
fig.update_traces(textposition="top center", marker=dict(size=8))
# add trendline
x_range = np.linspace(scatter_df["Growth_Pct"].min(), scatter_df["Growth_Pct"].max(), 100)
fig.add_scatter(x=x_range, y=intercept + slope * x_range, mode="lines",
                name=f"OLS (slope={slope:.2f})", line=dict(dash="dash", color="red"))
fig.update_layout(height=500, width=800)
fig.show()

print(f"Correlation: r = {r_value:.2f}, R² = {r_value**2:.2f}")
print(f"States with faster pandemic-era growth have proportionally higher MAPE.")

=== Year-by-Year Forecast Bias (Predicted − Actual) ===

  2020.0:  Mean Error:    -3.3  ( -0.4%)  |  MAPE:   0.8%  |  UNDER-predicted 36/51 states
  2021.0:  Mean Error:   -70.8  ( -8.9%)  |  MAPE:   8.9%  |  UNDER-predicted 51/51 states
  2022.0:  Mean Error:  -180.1  (-19.8%)  |  MAPE:  19.8%  |  UNDER-predicted 51/51 states
  2023.0:  Mean Error:  -223.8  (-23.8%)  |  MAPE:  23.8%  |  UNDER-predicted 51/51 states
  2024.0:  Mean Error:  -261.7  (-26.5%)  |  MAPE:  26.5%  |  UNDER-predicted 51/51 states

Overall: ARIMA underpredicted in 48/51 states on average.
The bias grows each year — the model missed the accelerating post-COVID surge.



Correlation: r = 0.79, R² = 0.63
States with faster pandemic-era growth have proportionally higher MAPE.


### 5d. Forecast Outlook: Predicted HPI Rankings in 2034

Using the fitted ARIMA models, we forecast HPI out to 2034 and rank states by predicted index value and growth rate.

In [15]:
# --- 5d. Forecast outlook: predicted HPI rankings in 2034 ---
print("=" * 65)
print("FORECAST OUTLOOK: PREDICTED HPI RANKINGS IN 2034")
print("=" * 65)

forecast_2034 = arima_forecast[arima_forecast["Year"] == 2034].copy()
latest_actual = df[df["Year"] == 2024][["Abbreviation", "HPI"]].rename(columns={"HPI": "HPI_2024"})
outlook = forecast_2034.merge(latest_actual, on="Abbreviation")
outlook["Growth_2024_2034_Pct"] = ((outlook["Predicted"] - outlook["HPI_2024"]) / outlook["HPI_2024"] * 100).round(1)

print("\nTop 10 states by predicted HPI in 2034:")
top = outlook.sort_values("Predicted", ascending=False).head(10)
for _, r in top.iterrows():
    print(f"  {r['Abbreviation']:2s} {r['State']:22s}  HPI: {r['Predicted']:8.1f}  (2024: {r['HPI_2024']:.1f}, +{r['Growth_2024_2034_Pct']:.1f}%)")

print("\nBottom 10 states by predicted HPI in 2034:")
bot = outlook.sort_values("Predicted").head(10)
for _, r in bot.iterrows():
    print(f"  {r['Abbreviation']:2s} {r['State']:22s}  HPI: {r['Predicted']:8.1f}  (2024: {r['HPI_2024']:.1f}, +{r['Growth_2024_2034_Pct']:.1f}%)")

print(f"\nHighest predicted growth:  {outlook.sort_values('Growth_2024_2034_Pct', ascending=False).iloc[0]['Abbreviation']} (+{outlook['Growth_2024_2034_Pct'].max():.1f}%)")
print(f"Lowest predicted growth:   {outlook.sort_values('Growth_2024_2034_Pct').iloc[0]['Abbreviation']} (+{outlook['Growth_2024_2034_Pct'].min():.1f}%)")
print(f"Median predicted growth:   +{outlook['Growth_2024_2034_Pct'].median():.1f}%")

FORECAST OUTLOOK: PREDICTED HPI RANKINGS IN 2034

Top 10 states by predicted HPI in 2034:
  CA California              HPI:   2500.4  (2024: 1955.5, +27.9%)
  DC District Of Columbia    HPI:   2406.0  (2024: 1975.5, +21.8%)
  CO Colorado                HPI:   2386.4  (2024: 1366.3, +74.7%)
  MT Montana                 HPI:   2371.1  (2024: 1262.2, +87.9%)
  WA Washington              HPI:   2256.6  (2024: 1836.6, +22.9%)
  MA Massachusetts           HPI:   2133.6  (2024: 1622.7, +31.5%)
  RI Rhode Island            HPI:   1757.6  (2024: 1344.9, +30.7%)
  NH New Hampshire           HPI:   1744.2  (2024: 1236.3, +41.1%)
  OR Oregon                  HPI:   1727.1  (2024: 1376.9, +25.4%)
  TN Tennessee               HPI:   1640.6  (2024: 858.5, +91.1%)

Bottom 10 states by predicted HPI in 2034:
  WV West Virginia           HPI:    522.3  (2024: 444.4, +17.5%)
  MS Mississippi             HPI:    607.7  (2024: 489.8, +24.1%)
  LA Louisiana               HPI:    694.6  (2024: 599.9, +15.8%)

### 5e. Confidence Interval Analysis

Wider confidence intervals indicate greater forecast uncertainty. Let's examine which states have the most and least certain forecasts.

In [16]:
# --- 5e. Confidence interval width analysis ---
print("=" * 65)
print("CONFIDENCE INTERVAL ANALYSIS")
print("=" * 65)
print("\nWider 95% CI = more uncertainty in the forecast.\n")

ci_analysis = arima_forecast.copy()
ci_analysis["CI_Width"] = ci_analysis["CI_Upper"] - ci_analysis["CI_Lower"]

# CI width in 2034 (last forecast year)
ci_2034 = ci_analysis[ci_analysis["Year"] == 2034].sort_values("CI_Width", ascending=False)
latest_actual = df[df["Year"] == 2024][["Abbreviation", "HPI"]].rename(columns={"HPI": "HPI_2024"})
ci_2034 = ci_2034.merge(latest_actual, on="Abbreviation")
ci_2034["CI_Width_Pct"] = (ci_2034["CI_Width"] / ci_2034["HPI_2024"] * 100).round(1)

print("States with widest 95% CI in 2034 (most uncertain):")
for _, r in ci_2034.head(10).iterrows():
    print(f"  {r['Abbreviation']:2s}  CI Width: {r['CI_Width']:8.1f}  ({r['CI_Width_Pct']:5.1f}% of 2024 HPI)")

print("\nStates with narrowest 95% CI in 2034 (most confident):")
for _, r in ci_2034.tail(10).iterrows():
    print(f"  {r['Abbreviation']:2s}  CI Width: {r['CI_Width']:8.1f}  ({r['CI_Width_Pct']:5.1f}% of 2024 HPI)")

# CI width growth over forecast horizon
print("\nAverage CI width by forecast year:")
for year in sorted(ci_analysis["Year"].unique()):
    avg_width = ci_analysis[ci_analysis["Year"] == year]["CI_Width"].mean()
    print(f"  {int(year)}: {avg_width:.1f}")

CONFIDENCE INTERVAL ANALYSIS

Wider 95% CI = more uncertainty in the forecast.

States with widest 95% CI in 2034 (most uncertain):
  CA  CI Width:   1573.8  ( 80.5% of 2024 HPI)
  MT  CI Width:   1358.8  (107.7% of 2024 HPI)
  CO  CI Width:   1254.1  ( 91.8% of 2024 HPI)
  WA  CI Width:   1237.5  ( 67.4% of 2024 HPI)
  RI  CI Width:   1096.8  ( 81.6% of 2024 HPI)
  MA  CI Width:   1089.0  ( 67.1% of 2024 HPI)
  NV  CI Width:   1044.5  ( 97.9% of 2024 HPI)
  NH  CI Width:    987.1  ( 79.8% of 2024 HPI)
  AZ  CI Width:    955.9  ( 85.7% of 2024 HPI)
  NJ  CI Width:    917.6  ( 71.5% of 2024 HPI)

States with narrowest 95% CI in 2034 (most confident):
  KS  CI Width:    370.0  ( 58.1% of 2024 HPI)
  OK  CI Width:    342.8  ( 52.6% of 2024 HPI)
  AK  CI Width:    331.1  ( 50.4% of 2024 HPI)
  ND  CI Width:    320.3  ( 46.2% of 2024 HPI)
  AR  CI Width:    316.1  ( 49.5% of 2024 HPI)
  AL  CI Width:    314.7  ( 51.6% of 2024 HPI)
  IA  CI Width:    265.2  ( 42.4% of 2024 HPI)
  LA  CI Widt

## 6. Key Takeaways

1. **ARIMA achieves a median MAPE of ~15.5%** on the 2020–2024 holdout — reasonable given this period includes the most volatile housing market in decades.

2. **Error is strongly correlated with pandemic-era housing boom intensity.** States like FL, NV, AZ, and ID that saw explosive price surges had the highest MAPE (>23%), while stable markets like DC, LA, and ND had MAPE under 8%.

3. **The model systematically underpredicts** in later holdout years (2022–2024), confirming that ARIMA — trained on pre-2020 trends — could not anticipate the accelerating post-COVID housing surge.

4. **Regional patterns are clear:** Sun Belt and Mountain West states are hardest to predict; Midwest and South Central states are the most accurate. This reflects fundamental differences in housing market dynamics: migration-driven booms vs. steady organic growth.

5. **Confidence intervals widen significantly** over the 10-year forecast horizon, reflecting growing uncertainty. By 2034, the average CI width is substantial — forecasts beyond 5 years should be treated as directional rather than precise.

6. **Limitations:** ARIMA is a univariate model — it cannot account for external factors like interest rates, migration patterns, housing supply, or policy changes. The 2025–2034 forecasts assume current trends continue, which is unlikely over a full decade.